# Lecture 2: Confidence intervals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture02/lecture02.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Newbold, Carlson and Thorne, sections 7.1 to 7.5 and 7.7.

This notebook follows the lecture slides. Every method is worked through in the same steps: the question, what we
know and want to find out, the formula, the calculation by hand (on the slides), the same calculation in Python,
and the answer for a manager. Parts 1 to 4 each end with an exercise. In an exercise you predict a result, run the
cell, change one value, and answer a question about what you see. You do not need to write any code. The last
section holds a summary table, the symbols and the Danish terms, so you can use the notebook as a reference
afterwards.

Run the setup cell first. It loads the RetailCorp data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # the study of 200 orders
population = pd.read_csv(url + "orders_population.csv")  # all 50,000 orders, used only in the demonstration

print(f"Sample: {len(sample)} orders. Population: {len(population):,} orders.")
sample.head()

## The case and the question

RetailCorp is considering a promise to customers that 95% of orders arrive within 48 hours, which means at most 5%
late. In Lecture 1 we knew the population and asked where a sample result would land. In practice the direction is
the opposite: RetailCorp has one sample, the study of 200 orders, and wants to know what it says about all orders.

**Given the one sample RetailCorp has, which values of the parameter are plausible?**

| RetailCorp question | Parameter | Part |
|---|---|---|
| How long does a delivery take on average? | μ | 1 and 2 |
| What share of orders arrives late, and does the promise hold? | P | 3 |
| How predictable are delivery times? | σ | 4 |
| How many orders should the next study check? | n | 5 |

In [ ]:
n = len(sample)
x_bar = sample["delivery_hours"].mean()
s = sample["delivery_hours"].std(ddof=1)
p_hat = sample["late"].mean()

print(f"n = {n}")
print(f"x_bar = {x_bar:.3f} hours")
print(f"s = {s:.3f} hours")
print(f"p_hat = {p_hat:.3f} ({sample['late'].sum()} late orders)")

## Part 1: The mean when σ is known

**Question.** How long does a delivery take on average, across all of RetailCorp's orders?

**What we know.** n = 200 orders and x̄ = 36.789 hours. For now we assume that σ = 7.819 hours is known. In
practice σ is rarely known; we start here because the formula follows directly from Lecture 1. Part 2 drops it.

**What we want.** A range of plausible values for μ, the average delivery time of all orders, with 95% confidence.

**Formula.** x̄ ± z<sub>α/2</sub> · σ/√n. Read z<sub>α/2</sub> as "z alpha-half": the value with α/2 above it
in the standard normal distribution, 1.960 for 95%. σ/√n is the standard error from Lecture 1, and
ME = z<sub>α/2</sub> · σ/√n is the margin of error.

`stats.norm.ppf` takes the area to the *left* of the value, so z<sub>α/2</sub> for 95% is `stats.norm.ppf(0.975)`.

In [ ]:
sigma = 7.819                  # assumed known
n = len(sample)
x_bar = sample["delivery_hours"].mean()
z = stats.norm.ppf(0.975)      # 0.025 above it
se = sigma / np.sqrt(n)
me = z * se

print(f"Standard error: {se:.4f}")
print(f"z: {z:.3f}")
print(f"Margin of error: {me:.3f}")
print(f"95% interval for mu: [{x_bar - me:.2f}, {x_bar + me:.2f}] hours")

**For the manager.** We estimate the average delivery time of all orders at 36.8 hours. With 95% confidence, it lies
between 35.7 and 37.9 hours.

### Exercise 1: changing the confidence level

**Predict.** The cell computes the interval at 95%. Will a 90% interval be wider or narrower? Write down your guess.

**Run** the cell.

In [ ]:
confidence = 0.95      # change to 0.90 in step 3, then to 0.99
alpha = 1 - confidence
z = stats.norm.ppf(1 - alpha / 2)
me = z * sigma / np.sqrt(n)

print(f"Confidence {confidence:.0%}: z = {z:.3f}, ME = {me:.2f}")
print(f"Interval: [{x_bar - me:.2f}, {x_bar + me:.2f}] hours")

**Change** `confidence = 0.95` to `0.90` and run the cell again. Then try `0.99`.

**Answer.** Why does a higher confidence level give a wider interval, and what does 99% cost?

## Part 2: The mean when σ is unknown

**What changes.** σ is unknown, so we use s = 8.140 hours in its place. This adds uncertainty, because s also
changes from sample to sample. If the population is normal, t = (x̄ − μ)/(s/√n) follows a t distribution with
n − 1 degrees of freedom. Its tails are wider than those of the standard normal, and it comes close to the standard
normal as n grows.

**Formula.** x̄ ± t<sub>n−1, α/2</sub> · s/√n. Read t<sub>n−1, α/2</sub> as "t with n minus one degrees of
freedom, alpha-half". The interval holds when the population is normal, or when n is large (central limit theorem).

The cell below shows the critical value for 95% for a few sample sizes.

In [ ]:
for n_ in [5, 10, 25, 200]:
    print(f"n = {n_:>3}: t = {stats.t.ppf(0.975, df=n_ - 1):.3f}")
print(f"standard normal: z = {stats.norm.ppf(0.975):.3f}")

In [ ]:
s = sample["delivery_hours"].std(ddof=1)     # ddof=1 divides by n - 1
t = stats.t.ppf(0.975, df=n - 1)
se = s / np.sqrt(n)
me = t * se

print(f"Standard error: {se:.4f}")
print(f"t with {n - 1} degrees of freedom: {t:.3f}")
print(f"Margin of error: {me:.3f}")
print(f"95% interval for mu: [{x_bar - me:.2f}, {x_bar + me:.2f}] hours")

**For the manager.** We estimate the average delivery time of all orders at 36.8 hours. With 95% confidence, it lies
between 35.7 and 37.9 hours. With 200 orders, using s in place of σ hardly changes the interval.

### Exercise 2: Copenhagen orders only

**Predict.** The cell computes the 95% interval for the 80 orders shipped from Copenhagen. Will it be wider or
narrower than the interval for all 200 orders? Write down your guess.

**Run** the cell.

In [ ]:
warehouse = "Copenhagen"      # change to "Aarhus" in step 3

x = sample.loc[sample["warehouse"] == warehouse, "delivery_hours"]
n_w = len(x)
t_w = stats.t.ppf(0.975, df=n_w - 1)
me_w = t_w * x.std(ddof=1) / np.sqrt(n_w)

print(f"{warehouse}: n = {n_w}, x_bar = {x.mean():.2f}, s = {x.std(ddof=1):.2f}")
print(f"t = {t_w:.3f}, ME = {me_w:.2f}")
print(f"Interval: [{x.mean() - me_w:.2f}, {x.mean() + me_w:.2f}] hours")

**Change** `warehouse = "Copenhagen"` to `"Aarhus"` and run the cell again.

**Answer.** Which two numbers in the output explain the difference in width?

### Demonstration: what 95% confidence means

This cell is run in the lecture. It draws 100 random samples of 25 orders from the population and computes a 95%
t interval from each. We can do this only because the population is simulated; in practice there is one sample.
Count how many of the intervals contain the true mean μ.

In [ ]:
n_demo = 25
reps = 100
rng = np.random.default_rng(3)
pop = population["delivery_hours"].to_numpy()
mu = pop.mean()
t_demo = stats.t.ppf(0.975, df=n_demo - 1)

lowers, uppers = [], []
for i in range(reps):
    x = rng.choice(pop, size=n_demo, replace=False)
    me_demo = t_demo * x.std(ddof=1) / np.sqrt(n_demo)
    lowers.append(x.mean() - me_demo)
    uppers.append(x.mean() + me_demo)

covered = 0
for i in range(reps):
    if lowers[i] < mu < uppers[i]:
        covered += 1
print(f"{covered} of {reps} intervals contain mu = {mu:.2f}")

for i in range(reps):
    color = "tab:blue" if lowers[i] < mu < uppers[i] else "tab:orange"
    plt.plot([i, i], [lowers[i], uppers[i]], color=color)
plt.axhline(mu, color="black")
plt.ylabel("Hours")
plt.title("100 intervals from samples of 25 orders")
plt.show()

The 95% describes the method: over many samples, about 95% of the intervals contain μ. For the one interval
RetailCorp has, μ is either inside or not, and we do not know which.

## Part 3: A proportion

**Question.** What share of all orders arrives late, and does the promise of at most 5% late hold?

**What we know.** n = 200 orders, 16 of them late, so p̂ = 0.080. p̂ is the mean of the 0/1 column `late`.

**What we want.** A range of plausible values for P, the share of late orders among all orders, with 95% confidence,
and whether these values lie below 0.05.

**Formula.** p̂ ± z<sub>α/2</sub> · √(p̂(1 − p̂)/n). The standard error uses p̂ in place of the unknown P.
Newbold's condition is nP(1 − P) > 5; P is unknown, so we check it with p̂.

In [ ]:
z = stats.norm.ppf(0.975)
p_hat = sample["late"].mean()
print(f"Condition n * p_hat * (1 - p_hat): {n * p_hat * (1 - p_hat):.2f}   (above 5)")

se_p = np.sqrt(p_hat * (1 - p_hat) / n)
me = z * se_p
print(f"Standard error: {se_p:.4f}")
print(f"Margin of error: {me:.4f}")
print(f"95% interval for P: [{p_hat - me:.3f}, {p_hat + me:.3f}]")

**For the manager.** Between about 4% and 12% of all orders arrive late (95% confidence). Our best estimate, 8%, is
above the 5% the promise allows, but values below 5% are also plausible. A study of 200 orders cannot settle whether
the promise holds.

### Exercise 3: late orders from Copenhagen

**Predict.** Will the 95% interval for the share of late Copenhagen orders contain 5%? Write down your guess.

**Run** the cell. It also prints the condition.

In [ ]:
warehouse = "Copenhagen"      # change to "Aarhus" in step 3

x = sample.loc[sample["warehouse"] == warehouse, "late"]
n_w = len(x)
p_w = x.mean()
me_w = z * np.sqrt(p_w * (1 - p_w) / n_w)

print(f"{warehouse}: n = {n_w}, late = {x.sum()}, p_hat = {p_w:.3f}")
print(f"Condition: {n_w * p_w * (1 - p_w):.1f}")
print(f"Interval: [{p_w - me_w:.3f}, {p_w + me_w:.3f}]")

**Change** `warehouse = "Copenhagen"` to `"Aarhus"` and run the cell again. Look at the condition.

**Answer.** What can RetailCorp say about Copenhagen orders and the promise?

## Part 4: A variance

**Question.** How predictable are delivery times? How large is the spread of single deliveries across all orders?

**What we know.** n = 200 orders, s² = 66.257 hours² and s = 8.140 hours.

**What we want.** A range of plausible values for σ² and σ, with 95% confidence.

**Formula.** If the population is normal, (n − 1)s²/σ² follows a chi-square distribution with n − 1 degrees of
freedom (Lecture 1). Newbold writes χ²<sub>n−1, α/2</sub> for the value with α/2 above it. The interval is

(n − 1)s² / χ²<sub>n−1, α/2</sub> < σ² < (n − 1)s² / χ²<sub>n−1, 1−α/2</sub>

The larger critical value gives the lower limit. Take square roots of both limits to get an interval for σ. The
interval holds only when the population is normal; a large n does not help. So we check the data first.

In [ ]:
plt.hist(sample["delivery_hours"], bins=20, edgecolor="white")
plt.xlabel("Delivery time (hours)")
plt.title("Delivery times of the 200 orders")
plt.show()
print(f"Skewness: {stats.skew(sample['delivery_hours']):.2f}")

In [ ]:
s2 = sample["delivery_hours"].var(ddof=1)
chi2_hi = stats.chi2.ppf(0.975, df=n - 1)   # the larger value, 0.025 above it
chi2_lo = stats.chi2.ppf(0.025, df=n - 1)   # the smaller value, 0.025 below it
lower = (n - 1) * s2 / chi2_hi
upper = (n - 1) * s2 / chi2_lo

print(f"(n - 1) s^2 = {(n - 1) * s2:.1f}")
print(f"Critical values: {chi2_hi:.2f} and {chi2_lo:.2f}")
print(f"95% interval for sigma^2: [{lower:.2f}, {upper:.2f}]")
print(f"95% interval for sigma: [{np.sqrt(lower):.2f}, {np.sqrt(upper):.2f}] hours")

**For the manager.** The standard deviation of single delivery times across all orders is between about 7.4 and 9.0
hours (95% confidence). This is the spread the 48-hour promise has to live with.

### Exercise 4: basket values

**Predict.** The cell computes the 95% interval for σ and draws a histogram. If you switch to basket values, will you
still get an interval? Should RetailCorp report it? Write down your guess.

**Run** the cell for delivery times.

In [ ]:
variable = "delivery_hours"      # change to "order_value_dkk" in step 3

x = sample[variable]
n_v = len(x)
lower_v = (n_v - 1) * x.var(ddof=1) / stats.chi2.ppf(0.975, df=n_v - 1)
upper_v = (n_v - 1) * x.var(ddof=1) / stats.chi2.ppf(0.025, df=n_v - 1)

print(f"95% interval for sigma: [{np.sqrt(lower_v):.2f}, {np.sqrt(upper_v):.2f}]")
print(f"Skewness: {stats.skew(x):.2f}")
plt.hist(x, bins=20, edgecolor="white")
plt.title(variable)
plt.show()

**Change** `variable = "delivery_hours"` to `"order_value_dkk"` and run the cell again.

**Answer.** Why should the interval for basket values not be reported?

## Part 5: Sample size

**Question.** How many orders must RetailCorp check to estimate the share of late orders within ±2 percentage
points, with 95% confidence?

**Formula.** Solving ME = z<sub>α/2</sub> · √(p̂(1 − p̂)/n) for n gives n = p̂(1 − p̂) · z<sub>α/2</sub>² / ME². Before
the study p̂ is unknown. p̂(1 − p̂) is at most 0.25, so Newbold uses n = 0.25 · z<sub>α/2</sub>² / ME², which gives
a sample large enough whatever P is. Round up to the next whole number. For a mean, n = z<sub>α/2</sub>² · σ² / ME²
(used in the workshop).

In [ ]:
z = stats.norm.ppf(0.975)
n_needed = 0.25 * z**2 / 0.02**2
print(f"n = {n_needed:.1f}, round up to {np.ceil(n_needed):.0f} orders")

**For the manager.** To estimate the share of late orders within ±2 percentage points, the next study needs about
2,400 orders, twelve times as many as the current study. With p̂ = 0.08 from this study in place of 0.25, about 707
orders would do, if P really is close to 8%.

## Reference

Every calculation from the slides is in the parts above. This section adds one check and the tables to revise from.

### Why the interval for basket values cannot be trusted

The chi-square interval assumes a normal population. Basket values are strongly skewed. Because the population is
simulated, we can check how often a 95% interval for the variance of basket values contains the true variance
σ² when RetailCorp draws samples of 200 orders. For delivery times, which are close to normal, the share is close
to 95%.

In [ ]:
n_check = 200
reps = 2000
rng = np.random.default_rng(4)
pop = population["order_value_dkk"].to_numpy()
sigma2 = pop.var()
chi2_hi = stats.chi2.ppf(0.975, df=n_check - 1)
chi2_lo = stats.chi2.ppf(0.025, df=n_check - 1)

covered = 0
for i in range(reps):
    x = rng.choice(pop, size=n_check, replace=False)
    s2_i = x.var(ddof=1)
    if (n_check - 1) * s2_i / chi2_hi < sigma2 < (n_check - 1) * s2_i / chi2_lo:
        covered += 1
print(f"Share of 95% intervals that contain the true variance: {covered / reps:.3f}")

### Four confidence intervals

Given the one sample RetailCorp has, which values of the parameter are plausible?

| Parameter | Interval | Distribution | Holds when | RetailCorp, 95% |
|---|---|---|---|---|
| μ, σ known (Part 1) | x̄ ± z<sub>α/2</sub> · σ/√n | Standard normal | Population normal, or n large | 35.71 to 37.87 hours |
| μ, σ unknown (Part 2) | x̄ ± t<sub>n−1, α/2</sub> · s/√n | t with n − 1 degrees of freedom | Population normal, or n large | 35.65 to 37.92 hours |
| P (Part 3) | p̂ ± z<sub>α/2</sub> · √(p̂(1 − p̂)/n) | Approximately normal | np̂(1 − p̂) > 5 | 4.2% to 11.8% |
| σ² (Part 4) | (n − 1)s²/χ²<sub>n−1, α/2</sub> to (n − 1)s²/χ²<sub>n−1, 1−α/2</sub> | Chi-square with n − 1 degrees of freedom | Population normal. A large n does not help. | σ: 7.41 to 9.03 hours |

Sample size (Part 5): n = 0.25 · z<sub>α/2</sub>² / ME² for a proportion and n = z<sub>α/2</sub>² · σ² / ME² for a
mean, rounded up.

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| θ, θ̂ | theta, theta-hat | Any parameter, and its point estimate |
| ± | plus or minus | The interval runs from the estimate minus ME to the estimate plus ME |
| α, 1 − α | alpha, one minus alpha | 1 − α is the confidence level; α = 0.05 for 95% |
| z<sub>α/2</sub> | z alpha-half | Value with α/2 above it in the standard normal distribution |
| t<sub>n−1, α/2</sub> | t with n minus one degrees of freedom, alpha-half | Value with α/2 above it in the t distribution |
| χ²<sub>n−1, α/2</sub> | chi-square with n minus one degrees of freedom, alpha-half | Value with α/2 above it in the chi-square distribution |
| ME | margin of error | Half the width of the interval |
| LCL, UCL | lower and upper confidence limit | The two ends of the interval |

### Python functions

| What | Python |
|---|---|
| z<sub>α/2</sub> | `stats.norm.ppf(1 - alpha / 2)` |
| t<sub>n−1, α/2</sub> | `stats.t.ppf(1 - alpha / 2, df=n - 1)` |
| χ²<sub>n−1, α/2</sub> and χ²<sub>n−1, 1−α/2</sub> | `stats.chi2.ppf(1 - alpha / 2, df=n - 1)` and `stats.chi2.ppf(alpha / 2, df=n - 1)` |
| Round up | `np.ceil(x)` |

`ppf` gives the value with a given probability *below* it, which is why the upper-tail values in Newbold's notation
use `1 - alpha / 2`.

### Danish terms

| English | Danish |
|---|---|
| point estimate, interval estimate | punktestimat, intervalestimat |
| confidence interval | konfidensinterval |
| confidence level | konfidensniveau |
| margin of error | fejlmargin |
| critical value | kritisk værdi |
| t distribution | t-fordeling |
| degrees of freedom | frihedsgrader |
| sample size | stikprøvestørrelse |